# Day 11 — Anatomy of a RAG You Can Evaluate

**Module 3 · RAG Evaluation**

Before evaluating a RAG system, we need a small RAG system to evaluate.

Today we build a simple pipeline:

```text
Question
   ↓
Retriever
   ↓
Retrieved Context
   ↓
LLM
   ↓
Answer

## 1. Setup

We use OpenAI for generation.

The concurrency limit is kept at `2` throughout the course to avoid unnecessary API pressure.


In [1]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found."

client = OpenAI()

print("OpenAI client ready.")

OpenAI client ready.


## 2. Our Small Knowledge Base

For learning purposes, we use five short documents.

Each document represents one topic.

Later, the retriever will search these documents and return the most relevant ones.

In [2]:
CORPUS = [
    {
        "title": "Tokens",
        "content": (
            "Large language models split text into tokens. "
            "Tokens are the basic units processed by an LLM, "
            "and both input and output are measured in tokens."
        ),
    },
    {
        "title": "Embeddings",
        "content": (
            "An embedding is a vector representation of text. "
            "Texts with similar meanings have vectors that are close together, "
            "which enables semantic search."
        ),
    },
    {
        "title": "Retrieval-Augmented Generation",
        "content": (
            "RAG grounds an LLM's answer in external documents. "
            "The system retrieves relevant information and provides it to the LLM "
            "as context, which can reduce hallucinations."
        ),
    },
    {
        "title": "Hallucination",
        "content": (
            "A hallucination is information generated by an LLM that is "
            "factually incorrect or unsupported by its sources."
        ),
    },
    {
        "title": "AI Agents",
        "content": (
            "An AI agent is an LLM given a goal, tools, and a loop that allows it "
            "to plan, take actions, observe results, and continue until the task is complete."
        ),
    },
]

print("Documents:", len(CORPUS))

Documents: 5


## 3. The Retriever

A retriever receives a question and selects documents that may contain the answer.

For now, we use a deliberately simple **keyword-overlap retriever**.

It is not a production retriever.

That is intentional: we want to clearly see what retrieval does before introducing embeddings and vector databases.

In [3]:
def retrieve(query: str, k: int = 2) -> list[str]:
    """Return the top-k documents using simple keyword overlap."""

    query_words = set(query.lower().split())

    scored_docs = []

    for doc in CORPUS:
        text = f"{doc['title']} {doc['content']}".lower()
        doc_words = set(text.split())

        score = len(query_words & doc_words)
        scored_docs.append((score, doc))

    scored_docs.sort(key=lambda item: item[0], reverse=True)

    return [
        doc["content"]
        for score, doc in scored_docs[:k]
    ]

## 4. Inspect Retrieval

Before generating an answer, we should inspect what the retriever actually returned.

This distinction is critical for RAG evaluation:

> **The answer can only be as good as the context available to the generator.**

In [4]:
question = "How does RAG reduce hallucination?"

retrieved_context = retrieve(question, k=2)

print("Question:", question)
print("\nRetrieved context:")

for i, passage in enumerate(retrieved_context, start=1):
    print(f"\n--- Passage {i} ---")
    print(passage)

Question: How does RAG reduce hallucination?

Retrieved context:

--- Passage 1 ---
RAG grounds an LLM's answer in external documents. The system retrieves relevant information and provides it to the LLM as context, which can reduce hallucinations.

--- Passage 2 ---
Large language models split text into tokens. Tokens are the basic units processed by an LLM, and both input and output are measured in tokens.


## 5. The Generator

The generator receives:

- the user's question
- the retrieved context

It should answer using the provided context.

If the context does not contain the answer, it should say that the information is unavailable rather than inventing an answer.

In [5]:
SYSTEM_PROMPT = """
Answer the question using only the provided context.

If the context does not contain enough information to answer,
say that you do not have enough information.

Be concise and answer in 1-2 sentences.
""".strip()


def generate_answer(question: str, context: list[str]) -> str:
    context_text = "\n\n".join(context)

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": (
                    f"Context:\n{context_text}\n\n"
                    f"Question: {question}"
                ),
            },
        ],
    )

    return response.choices[0].message.content.strip()

## 6. Combine the Pipeline

Now we combine retrieval and generation into one function.

The function returns **both the answer and the retrieved context**.

Returning the context is important because later our evaluation will need to inspect both.

In [6]:
def ask_rag(question: str, k: int = 2) -> tuple[str, list[str]]:
    context = retrieve(question, k=k)
    answer = generate_answer(question, context)

    return answer, context

## 7. Run the RAG

Let's see the complete pipeline in action.

In [7]:
questions = [
    "What is a token in an LLM?",
    "How does RAG reduce hallucination?",
    "What is an AI agent?",
]

for question in questions:
    answer, context = ask_rag(question)

    print("=" * 70)
    print("Question:", question)
    print("Answer:", answer)
    print("Retrieved passages:", len(context))

Question: What is a token in an LLM?
Answer: The provided context does not contain information about what a token is in an LLM.
Retrieved passages: 2
Question: How does RAG reduce hallucination?
Answer: RAG reduces hallucination by grounding an LLM's answer in relevant external documents retrieved and provided as context, ensuring the response is based on accurate information.
Retrieved passages: 2
Question: What is an AI agent?
Answer: An AI agent is a large language model (LLM) given a goal, tools, and a loop that allows it to plan, take actions, observe results, and continue until the task is complete.
Retrieved passages: 2


## 8. What Can Go Wrong?

A RAG system has multiple failure points:

```text
Documents
    ↓
 Retrieval
    ↓
 Context
    ↓
 Generation
    ↓
  Answer

## 9. The Evaluation Data We Need

For a RAG evaluation case, we eventually want to capture:

| Field | Meaning |
|---|---|
| `input` | User's question |
| `retrieval_context` | What the retriever returned |
| `actual_output` | What the LLM answered |
| `expected_output` | The reference answer, when available |

These fields allow us to ask different questions:

- **Did we retrieve the right information?**
- **Was the answer grounded in that information?**
- **Did the answer actually answer the question?**
- **Was the answer correct?**

Starting tomorrow, we begin evaluating these failure points with DeepEval.

# Day 11 — Key Takeaways

Today we built a deliberately simple RAG system.

The important concepts are:

1. A RAG system has **retrieval and generation** stages.
2. The retriever determines what context the LLM gets.
3. The generator produces the final answer from that context.
4. We must keep the **retrieved context** available for evaluation.
5. Different parts of the pipeline can fail independently.
